# Step 10: Mirror Lossless Branch to Main & Remove Test Split (CrossVQA)

This notebook mirrors the high-fidelity shards from the **`lossless`** branch to the **`main`** branch across both canonical repositories, while intentionally **pruning the blind `test` split**:
- **`gamusa/textvqa-canonical`**
- **`gamusa/docvqa-canonical`**

### Why Prune the Test Parquet Split from `main`?
1. **Empty Ground-Truth Labels:** As standard in benchmark challenges (Meta AI TextVQA / CVC DocVQA), the test split annotations are permanently withheld (`answers = []`), making them unusable for local offline scoring or error analysis.
2. **Bandwidth & Storage Optimization:** Removing test shards frees up ~3.5 GB of download bandwidth and disk footprint for downstream researchers fine-tuning and evaluating models locally.
3. **Zero-Download Server-Side Execution:** Leverages Hugging Face Hub's **`CommitOperationCopy`** to copy LFS pointers server-side in a single atomic commit per repo, eliminating the need to download or re-upload 17 GB of data.

In [ ]:
# 1. Environment Setup & Authentication
!pip install -q --upgrade huggingface_hub pyyaml datasets brotli

import os
import re
import yaml
from typing import List, Dict
from huggingface_hub import (
    HfApi,
    CommitOperationCopy,
    CommitOperationDelete,
    CommitOperationAdd,
    hf_hub_download,
    login
)

# Dual-environment Token Auth (Colab userdata + local env fallback)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("Please configure HF_TOKEN in Colab Secrets or environment variables with write access!")

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
user_info = api.whoami()
print(f"âœ“ Authenticated as: {user_info['name']}")

In [ ]:
# 2. Metadata Sanitizer: Clean YAML Frontmatter to Remove Test Split
def sanitize_readme_for_main(raw_readme: str) -> str:
    """
    Extracts YAML frontmatter from README.md, removes the test split from
    the configs/data_files section, and appends a clear benchmark note.
    """
    # Split YAML frontmatter from Markdown body
    parts = raw_readme.split("---\n")
    if len(parts) >= 3:
        yaml_content = parts[1]
        body_content = "---\n".join(parts[2:])
        
        meta = yaml.safe_load(yaml_content) or {}
        
        # Remove 'test' split from configs -> data_files
        if "configs" in meta and isinstance(meta["configs"], list):
            for cfg in meta["configs"]:
                if "data_files" in cfg and isinstance(cfg["data_files"], list):
                    cfg["data_files"] = [
                        item for item in cfg["data_files"]
                        if item.get("split") != "test"
                    ]
        
        cleaned_yaml = yaml.dump(meta, sort_keys=False, default_flow_style=False)
        
        # Append documentation note regarding test split removal
        note = (
            "\n\n> **Note on Benchmark Splits (main branch):** This repository contains curated `train` and `validation` splits "
            "for reproducible cross-domain VQA research. The blind challenge `test` split (which has withheld ground-truth labels) "
            "has been removed from this branch. For the complete unpruned benchmark, refer to the `lossless` branch.\n"
        )
        
        return f"---\n{cleaned_yaml}---\n{body_content.strip()}{note}"
    
    return raw_readme

In [ ]:
# 3. Atomic Server-Side Mirror & Prune Pipeline
def mirror_lossless_to_main(repo_id: str, api: HfApi):
    """
    Copies all train and validation Parquet shards from 'lossless' to 'main'
    using server-side LFS copy (CommitOperationCopy), prunes all test shards and old files,
    and updates the dataset card YAML frontmatter.
    """
    print(f"\n==================================================")
    print(f"Processing Repository: {repo_id}")
    print(f"==================================================")
    
    # 1. Inspect source ('lossless') and target ('main') branches
    lossless_files = set(api.list_repo_files(repo_id=repo_id, repo_type="dataset", revision="lossless"))
    main_files = set(api.list_repo_files(repo_id=repo_id, repo_type="dataset", revision="main"))
    
    print(f"Files on branch 'lossless': {len(lossless_files)}")
    print(f"Files on branch 'main' (pre-update): {len(main_files)}")
    
    # 2. Select files to mirror (Train + Validation Parquet shards)
    files_to_copy = [
        f for f in lossless_files
        if (f.startswith("data/train-") or f.startswith("data/validation-"))
    ]
    test_files_omitted = [f for f in lossless_files if f.startswith("data/test-")]
    print(f"âœ“ Shards to mirror (train + val): {len(files_to_copy)}")
    print(f"âœ— Test shards pruned from mirror: {len(test_files_omitted)}")
    
    # 3. Identify files to delete from main
    # (Delete all existing data/ parquet files currently in main to prevent obsolete shards lingering)
    files_to_delete = [f for f in main_files if f.startswith("data/")]
    
    # 4. Prepare README.md with sanitized frontmatter
    local_readme_path = hf_hub_download(repo_id, "README.md", repo_type="dataset", revision="lossless")
    with open(local_readme_path, "r", encoding="utf-8") as f:
        raw_readme = f.read()
    sanitized_readme = sanitize_readme_for_main(raw_readme)
    
    # 5. Build atomic commit operations
    operations = []
    
    # A. Delete obsolete data files on main
    for f in files_to_delete:
        operations.append(CommitOperationDelete(path_in_repo=f))
        
    # B. Server-side copy from branch 'lossless' to 'main'
    for f in files_to_copy:
        operations.append(
            CommitOperationCopy(
                src_path_in_repo=f,
                path_in_repo=f,
                src_revision="lossless"
            )
        )
        
    # C. Add sanitized README.md
    operations.append(
        CommitOperationAdd(
            path_in_repo="README.md",
            path_or_fileobj=sanitized_readme.encode("utf-8")
        )
    )
    
    print(f"\nSubmitting atomic commit: {len(operations)} operations ({len(files_to_delete)} deletes, {len(files_to_copy)} copies, 1 README update)...")
    
    commit_info = api.create_commit(
        repo_id=repo_id,
        repo_type="dataset",
        revision="main",
        operations=operations,
        commit_message="Mirror lossless branch to main (train + validation only; remove test split)"
    )
    
    print(f"âœ“ Commit successful! Commit URL: {commit_info.commit_url}")

In [ ]:
# 4. Execute Mirroring for Both CrossVQA Repositories
TARGET_REPOS = [
    "gamusa/textvqa-canonical",
    "gamusa/docvqa-canonical"
]

for repo_id in TARGET_REPOS:
    mirror_lossless_to_main(repo_id, api)

In [21]:
# 5. Post-Execution Verification: File Counts & Streaming Check
# Fix httpx2 / brotlicffi incompatibility when decoding compressed responses
try:
    import httpx2._decoders
    class SafeBrotliDecoder(httpx2._decoders.ContentDecoder):
        def __init__(self):
            try:
                import brotli
                self.decompressor = brotli.Decompressor()
            except Exception:
                import brotlicffi as brotli
                self.decompressor = brotli.Decompressor()
        def decode(self, data: bytes):
            if not data:
                return
            if hasattr(self.decompressor, 'process'):
                yield self.decompressor.process(data)
            else:
                yield self.decompressor.decompress(data)
        def flush(self):
            yield from ()
    httpx2._decoders.BrotliDecoder = SafeBrotliDecoder
except Exception:
    pass

from datasets import load_dataset

print("\n==================================================")
print("POST-EXECUTION VERIFICATION ON BRANCH 'main'")
print("==================================================")

for repo_id in TARGET_REPOS:
    files = api.list_repo_files(repo_id=repo_id, repo_type="dataset", revision="main")
    train_shards = [f for f in files if f.startswith("data/train-")]
    val_shards = [f for f in files if f.startswith("data/validation-")]
    test_shards = [f for f in files if f.startswith("data/test-")]
    
    print(f"\n[{repo_id}]")
    print(f"  Total files on 'main': {len(files)}")
    print(f"  Train shards: {len(train_shards)}")
    print(f"  Validation shards: {len(val_shards)}")
    print(f"  Test shards (should be 0): {len(test_shards)}")
    assert len(test_shards) == 0, f"Error: test shards still exist in {repo_id}!"
    
    # Live Streaming Sanity Check
    print(f"  Testing live stream on branch 'main' [split: validation]...")
    ds = load_dataset(repo_id, revision="main", split="validation", streaming=True)
    first_item = next(iter(ds))
    print(f"  âœ“ Successfully streamed sample_id: {first_item['sample_id']}")
    print(f"    Question: {first_item['question']}")
    print(f"    Answer: {first_item['canonical_answer']}")

print("\nâœ“ ALL REPOSITORIES ON 'main' SUCCESSFULLY MIRRORED AND VERIFIED!")


POST-EXECUTION VERIFICATION ON BRANCH 'main'

[gamusa/textvqa-canonical]
  Total files on 'main': 18
  Train shards: 14
  Validation shards: 2
  Test shards (should be 0): 0
  Testing live stream on branch 'main' [split: validation]...
  âœ“ Successfully streamed sample_id: textvqa_34602
    Question: what is the brand of this camera?
    Answer: dakota

[gamusa/docvqa-canonical]
  Total files on 'main': 59
  Train shards: 50
  Validation shards: 7
  Test shards (should be 0): 0
  Testing live stream on branch 'main' [split: validation]...


Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

  âœ“ Successfully streamed sample_id: docvqa_49153
    Question: What is the ‘actual’ value per 1000, during the year 1975?
    Answer: 0.28

âœ“ ALL REPOSITORIES ON 'main' SUCCESSFULLY MIRRORED AND VERIFIED!
